# TISER Dataset Chunker for Manual Translation

This notebook is designed to support a **human-in-the-loop translation process**
for the TISER dataset.

The goal is to:
- Load a large JSON dataset (train or test)
- Split it into manageable chunks
- Display **one chunk at a time**
- Allow manual translation using ChatGPT / Gemini
- Avoid confusion or loss of alignment between chunks

Each execution of the main cell will output **exactly one chunk**.

## Configuration

Set:
- the input dataset path
- the chunk size
- the progress file used to remember which chunk was last shown

The progress file ensures that you can safely stop and resume the process
without reprocessing the same examples.

In [48]:
import json
from pathlib import Path

# ===== USER CONFIG =====
DATASET_PATH = Path("/Users/usermastro/Desktop/Primo_Semestre_2526/DNLP/Project/TISER_repo/tools/datasets/TISER_train_10pct.json")   # change to train or test
CHUNK_SIZE = 20                                    # examples per chunk
PROGRESS_FILE = Path(".translation_progress.json") # internal state
# =======================

## Load Dataset

The dataset is expected to be a JSON list where each element corresponds
to one TISER example.

No modification is applied at this stage.

In [7]:
with DATASET_PATH.open("r", encoding="utf-8") as f:
    dataset = []
    with DATASET_PATH.open("r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                dataset.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON on line {line_num}: {e}") from e

total_examples = len(dataset)
total_chunks = (total_examples + CHUNK_SIZE - 1) // CHUNK_SIZE

print(f"Loaded dataset: {DATASET_PATH}")
print(f"Total examples: {total_examples}")
print(f"Chunk size: {CHUNK_SIZE}")
print(f"Total chunks: {total_chunks}")

Loaded dataset: /Users/usermastro/Desktop/Primo_Semestre_2526/DNLP/Project/TISER_repo/tools/datasets/TISER_train_10pct.json
Total examples: 5370
Chunk size: 20
Total chunks: 269


## One-time: store the translated TISER prompt

All samples share the same `prompt`, so we translate it once and reuse it.

Create a file (or paste in a cell) that contains the **Italian translation** of the TISER prompt.
The pipeline will re-insert this prompt into every translated example during the append step.

In [17]:
from pathlib import Path

# Store the translated prompt here (one time)
PROMPT_IT_PATH = Path("prompt_it.txt")

if not PROMPT_IT_PATH.exists():
    raise FileNotFoundError(
        f"Missing {PROMPT_IT_PATH}. Create it and paste the translated prompt there."
    )

prompt_it = PROMPT_IT_PATH.read_text(encoding="utf-8").strip()
print("Loaded prompt_it. Characters:", len(prompt_it))

Loaded prompt_it. Characters: 2126


---

---

## Export Next Chunk for Human-in-the-Loop Translation

This cell performs the **core operational step** of the human-in-the-loop translation pipeline.

Specifically, each execution:

1. **Reads the current progress state** to determine which chunk should be processed next.
2. **Extracts the next chunk** of the dataset based on the configured chunk size.
3. **Removes the `prompt` field** from each example, since the TISER prompt is fixed and translated only once.
4. **Exports two artifacts**:
   - A **pure JSON file** (`chunk_xxxx.json`) containing only the dataset examples, which remains machine-readable and can be validated programmatically.
   - A **ChatGPT/Gemini-ready text file** (`chunk_xxxx_for_chat.txt`) that contains:
     - The full translation prompt (loaded from an external file),
     - Followed by the JSON chunk to be translated.

The text file is intentionally **not valid JSON**, as it embeds natural language instructions before the JSON payload.
This design allows the user to **copy and paste the entire file content directly into ChatGPT or Gemini**
without manual assembly of the prompt and the data.

After exporting the files, the cell **updates the progress file** so that subsequent executions automatically move to the next chunk.
This ensures a deterministic, resumable, and error-resistant translation workflow.

In [107]:
import json
import re
from pathlib import Path

# =======================
# PATHS
# =======================
OUT_DIR = Path("chunks_to_translate")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PROMPT_PATH = Path("prompt_for_CHAT.txt")
if not PROMPT_PATH.exists():
    raise FileNotFoundError(f"Prompt file not found: {PROMPT_PATH.resolve()}")

PROMPT_TEXT = PROMPT_PATH.read_text(encoding="utf-8").strip()

# =======================
# HELPERS — SAFE UNICODE FIX
# =======================

_unicode_pattern = re.compile(r'\\u([0-9a-fA-F]{4})')

def _decode_unicode_only(s: str) -> str:
    """
    Decode ONLY \\uXXXX sequences into real Unicode characters.
    Leaves \\n, \\t, \\\\ etc. untouched.
    """

    # Fix missing-backslash forms: u00e9 -> \u00e9
    s = re.sub(r'(?<!\\)\bu00([0-9a-fA-F]{2})\b', r'\\u00\1', s, flags=re.IGNORECASE)

    # Replace \uXXXX selectively
    def _replace(match):
        codepoint = int(match.group(1), 16)
        return chr(codepoint)

    return _unicode_pattern.sub(_replace, s)


def make_chatgpt_txt(prompt_text: str, json_obj) -> str:
    """
    Build prompt + JSON for ChatGPT, decoding ONLY unicode letters.
    Keeps \\n as \\n.
    """
    json_text = json.dumps(json_obj, ensure_ascii=False, indent=2)
    full = prompt_text + "\n\n" + json_text
    return _decode_unicode_only(full)

# =======================
# LOAD PROGRESS
# =======================
if PROGRESS_FILE.exists():
    with PROGRESS_FILE.open("r", encoding="utf-8") as f:
        progress = json.load(f)
    current_chunk_idx = int(progress.get("current_chunk", 0))
else:
    current_chunk_idx = 0

print(f"Current chunk index: {current_chunk_idx}/{total_chunks}")

# =======================
# EXPORT NEXT CHUNK
# =======================
if current_chunk_idx >= total_chunks:
    print("All chunks have already been processed.")
else:
    start = current_chunk_idx * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, total_examples)
    chunk = dataset[start:end]

    # Remove "prompt"
    chunk_wo_prompt = []
    for ex in chunk:
        ex_dict = dict(ex)
        ex_dict.pop("prompt", None)
        chunk_wo_prompt.append(ex_dict)

    chunk_number = current_chunk_idx + 1

    # ---- JSON BACKUP ----
    json_path = OUT_DIR / f"chunk_{chunk_number:04d}.json"
    json_path.write_text(
        json.dumps(chunk_wo_prompt, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    # ---- TXT FOR LLM ----
    txt_path = OUT_DIR / f"chunk_{chunk_number:04d}_for_LLM.txt"
    chat_txt = make_chatgpt_txt(PROMPT_TEXT, chunk_wo_prompt)
    txt_path.write_text(chat_txt, encoding="utf-8")

    # Update progress
    with PROGRESS_FILE.open("w", encoding="utf-8") as f:
        json.dump({"current_chunk": chunk_number}, f, ensure_ascii=False, indent=2)

    print(f"Saved JSON chunk: {json_path.resolve()}")
    print(f"Saved ChatGPT-ready file (\\n preserved): {txt_path.resolve()}")
    print(f"Next chunk will be: {chunk_number + 1:04d}")

Current chunk index: 21/269
Saved JSON chunk: /Users/usermastro/Desktop/Primo_Semestre_2526/DNLP/Project/TISER_repo/tools/translation/chunks_to_translate/chunk_0022.json
Saved ChatGPT-ready file (\n preserved): /Users/usermastro/Desktop/Primo_Semestre_2526/DNLP/Project/TISER_repo/tools/translation/chunks_to_translate/chunk_0022_for_LLM.txt
Next chunk will be: 0023


---

## Validate and Sanitize the Last Translated Chunk

This cell performs a **full validation and cleanup pass** on the most recently translated chunk,
automatically inferred from the progress file.

It is designed to catch and fix the most common issues that arise when JSON chunks are translated
manually using ChatGPT or Gemini.

Specifically, the cell executes the following steps:

### 1. Identify the Last Generated Chunk
- Reads the progress file to determine the last exported chunk index.
- Locates the corresponding file `chunks_to_translate/chunk_XXXX.json`.
- Fails immediately if the progress file or chunk file is missing.

### 2. Sanitize Common Copy/Paste Artifacts
- Replaces smart quotes and typographic characters with standard ASCII equivalents.
- Normalizes ellipses and other problematic Unicode punctuation.
- This step is necessary because LLM outputs often introduce characters that break JSON parsing.

### 3. Parse and Rewrite Valid JSON
- Attempts to parse the sanitized text using `json.loads`.
- If parsing fails, prints:
  - the exact JSON error,
  - line and column information,
  - a small surrounding context window to quickly identify the issue.
- If parsing succeeds, rewrites the file as clean, pretty-printed UTF-8 JSON.
  This guarantees the chunk is stable and machine-readable.

### 4. Reconstruct the English Reference Chunk
- Rebuilds the corresponding English chunk directly from the original dataset,
  using the same chunk indices.
- Removes the `prompt` field to match the translated chunk structure.
- This reconstructed chunk acts as a ground-truth reference for alignment checks.

### 5. Structural and Alignment Validation
The cell verifies that:
- The translated chunk is a JSON array.
- Every element is a JSON object.
- The number of examples matches between English and Italian.
- The `question_id` order is identical (no shuffling or missing samples).
- All required keys are present in each translated example:
  `dataset_name`, `question_id`, `question`, `answer`, `output`, `context`.

### 6. Final Outcome
- If all checks pass, the cell prints a success message.
- At this point, the chunk is guaranteed to be:
  - valid JSON,
  - structurally aligned with the original dataset,
  - safe to append to the final Italian train or test file.

In [108]:
import json
import re
from pathlib import Path

OUT_DIR = Path("chunks_to_translate")

# -----------------------------
# 0) Deduce last chunk from progress
# -----------------------------
if not PROGRESS_FILE.exists():
    raise FileNotFoundError(f"Progress file not found: {PROGRESS_FILE.resolve()}")

with PROGRESS_FILE.open("r", encoding="utf-8") as f:
    progress = json.load(f)

last_chunk_number = int(progress.get("current_chunk", 0))  # 1-based
if last_chunk_number <= 0:
    raise ValueError(
        f"progress['current_chunk'] is {last_chunk_number}. "
        "Run the export cell first to generate a chunk."
    )

chunk_path = OUT_DIR / f"chunk_{last_chunk_number:04d}.json"
if not chunk_path.exists():
    raise FileNotFoundError(f"Chunk file not found: {chunk_path.resolve()}")

print("Last chunk number:", last_chunk_number)
print("Chunk path:", chunk_path.resolve())

# -----------------------------
# 1) Read file + sanitize quotes/ellipsis + fix broken unicode escapes
# -----------------------------
text = chunk_path.read_text(encoding="utf-8")

# 1.1 Normalize typical problematic characters (smart quotes, ellipsis)
text = (text
    .replace("“", "\"")
    .replace("”", "\"")
    .replace("„", "\"")
    .replace("«", "\"")
    .replace("»", "\"")
    .replace("‘", "'")
    .replace("’", "'")
    .replace("…", "...")
)

# 1.2 Fix broken unicode escapes introduced by model outputs:
#     "Alaquu00e0s" or "Garc\u00eda" missing the backslash -> "Alaqu\u00e0s"
#     We only fix patterns u00XX not already preceded by a backslash.
text = re.sub(r'(?<!\\)u00([0-9a-fA-F]{2})', r'\\u00\1', text)

# 1.3 Optional: collapse accidental multiple dots ".." -> "."
#     (harmless formatting drift)
text = re.sub(r'\.\.+', '.', text)

# -----------------------------
# 2) Parse JSON (hard fail with context if invalid)
# -----------------------------
try:
    chunk_it_wo_prompt = json.loads(text)
except json.JSONDecodeError as e:
    print("❌ Still invalid JSON after sanitization + unicode fix.")
    print("Error:", e)
    print("Line:", e.lineno, "Col:", e.colno, "Pos:", e.pos)

    lines = text.splitlines()
    start_ctx = max(e.lineno - 3, 0)
    end_ctx = min(e.lineno + 2, len(lines))

    print("\n--- Context ---")
    for i in range(start_ctx, end_ctx):
        marker = ">>" if (i + 1) == e.lineno else "  "
        print(f"{marker} {i+1:04d}: {lines[i]}")
    raise

# -----------------------------
# 3) Rewrite clean, stable JSON back to file (canonical UTF-8)
# -----------------------------
chunk_path.write_text(
    json.dumps(chunk_it_wo_prompt, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
print("✅ Sanitized + rewritten valid JSON:", chunk_path.resolve())

# -----------------------------
# 4) Type checks for IT chunk
# -----------------------------
if not isinstance(chunk_it_wo_prompt, list):
    raise ValueError(f"{chunk_path.name} is not a JSON array.")

for i, item in enumerate(chunk_it_wo_prompt):
    if not isinstance(item, dict):
        raise ValueError(f"{chunk_path.name}: element {i} is not a JSON object.")

# -----------------------------
# 5) Reconstruct EN reference chunk (without prompt)
# -----------------------------
start = (last_chunk_number - 1) * CHUNK_SIZE
end = min(start + CHUNK_SIZE, total_examples)
chunk_en = dataset[start:end]

chunk_en_wo_prompt = []
for ex in chunk_en:
    ex_dict = dict(ex)
    ex_dict.pop("prompt", None)
    chunk_en_wo_prompt.append(ex_dict)

print("EN reference range:", start, "→", end - 1, "count =", len(chunk_en_wo_prompt))
print("IT translated count:", len(chunk_it_wo_prompt))

# -----------------------------
# 6) Validation: length, IDs, required keys
# -----------------------------
# 6.1 same length
if len(chunk_it_wo_prompt) != len(chunk_en_wo_prompt):
    raise ValueError(
        f"Different number of examples: EN={len(chunk_en_wo_prompt)} IT={len(chunk_it_wo_prompt)}"
    )

# 6.2 same question_id order
ids_en = [ex.get("question_id") for ex in chunk_en_wo_prompt]
ids_it = [ex.get("question_id") for ex in chunk_it_wo_prompt]

if ids_en != ids_it:
    for i, (a, b) in enumerate(zip(ids_en, ids_it)):
        if a != b:
            raise ValueError(f"question_id mismatch at index {i}: EN={a} IT={b}")
    raise ValueError("question_id mismatch (order differs).")

# 6.3 required keys present
required_keys = {"dataset_name", "question_id", "question", "answer", "output", "context"}
for i, ex in enumerate(chunk_it_wo_prompt):
    missing = required_keys - set(ex.keys())
    if missing:
        raise ValueError(
            f"Missing keys at index {i} (question_id={ex.get('question_id')}): {missing}"
        )

print("✅ Validation passed.")
print("Examples:", len(chunk_it_wo_prompt))

Last chunk number: 22
Chunk path: /Users/usermastro/Desktop/Primo_Semestre_2526/DNLP/Project/TISER_repo/tools/translation/chunks_to_translate/chunk_0022.json
✅ Sanitized + rewritten valid JSON: /Users/usermastro/Desktop/Primo_Semestre_2526/DNLP/Project/TISER_repo/tools/translation/chunks_to_translate/chunk_0022.json
EN reference range: 420 → 439 count = 20
IT translated count: 20
✅ Validation passed.
Examples: 20


---

# Reinsert the translated prompt and append to the cumulative Italian dataset

We now:
- add `prompt: <prompt_it>` back into each example
- append them to a growing output file (JSONL is recommended)
- this creates a single Italian dataset file that grows chunk by chunk

Why JSONL:
- safe incremental append (no need to re-write a giant JSON array each time)
- robust against partial writes and large file sizes

In [109]:
OUT_IT_JSONL = Path("TISER_it.jsonl")  # change name if you want

def reinsert_prompt(examples, prompt_text: str):
    out = []
    for ex in examples:
        d = dict(ex)
        d["prompt"] = prompt_text
        out.append(d)
    return out

def append_jsonl(path: Path, examples):
    with path.open("a", encoding="utf-8") as f:
        for ex in examples:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")

chunk_it_full = reinsert_prompt(chunk_it_wo_prompt, prompt_it)
append_jsonl(OUT_IT_JSONL, chunk_it_full)

print(f"Appended {len(chunk_it_full)} examples to: {OUT_IT_JSONL.resolve()}")

Appended 20 examples to: /Users/usermastro/Desktop/Primo_Semestre_2526/DNLP/Project/TISER_repo/tools/translation/TISER_it.jsonl


---

---

## Notes on Translation

When translating a chunk:

- **Preserve the JSON structure exactly**
- Do NOT rename keys
- Do NOT remove or add fields
- Translate only natural language content:
  - question
  - context
  - reasoning
  - timeline
  - reflection
  - answer
- Keep all special tags unchanged:
  `<reasoning>`, `<timeline>`, `<reflection>`, `<answer>`

Once translated, the chunk can be appended to the target dataset
using the companion script.